In [59]:
!pip install lightgbm

In [60]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import classification_report, roc_auc_score
from sklearn.metrics import precision_recall_fscore_support

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [61]:
policy_df = pd.read_csv('../data/insurance_policyholder_churn_synthetic.csv')

policy_df.head()
policy_df.info()
policy_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   customer_id                  50000 non-null  int64  
 1   as_of_date                   50000 non-null  object 
 2   region_name                  50000 non-null  object 
 3   age                          50000 non-null  int64  
 4   age_band                     50000 non-null  object 
 5   marital_status               50000 non-null  object 
 6   customer_tenure_months       50000 non-null  int64  
 7   multi_policy_flag            50000 non-null  int64  
 8   num_policies                 50000 non-null  int64  
 9   policy_type                  50000 non-null  object 
 10  renewal_month                50000 non-null  int64  
 11  current_premium              50000 non-null  float64
 12  premium_last_year            50000 non-null  float64
 13  premium_change_p

,customer_id,age,customer_tenure_months,multi_policy_flag,num_policies,renewal_month,current_premium,premium_last_year,premium_change_pct,num_price_increases_last_3y,...,payout_ratio_12m,avg_settlement_time_days,days_since_last_claim,num_contacts_12m,complaint_flag,complaint_resolution_days,quote_requested_flag,coverage_downgrade_flag,churn_flag,churn_probability_true
count,50000.000000,50000.000000,50000.00000,50000.000000,50000.000000,50000.00000,50000.000000,50000.000000,50000.000000,50000.000000,...,50000.000000,50000.00000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,25000.500000,51.461700,90.36882,0.318640,1.638760,6.49762,1047.882399,994.319903,0.060795,1.669360,...,0.774165,18.17572,971.374400,1.285900,0.040440,0.493060,0.108480,0.098520,0.301660,0.303564
std,14433.901067,19.644405,51.83518,0.465954,1.041848,3.43503,455.932852,440.611049,0.080826,0.818957,...,0.039917,7.94916,501.320714,1.156213,0.196991,2.673779,0.310989,0.298019,0.458983,0.232942
min,1.000000,18.000000,1.00000,0.000000,1.000000,1.00000,120.000000,92.740000,-0.160200,0.000000,...,0.550000,1.00000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.018915
25%,12500.750000,34.000000,45.00000,0.000000,1.000000,4.00000,745.707500,701.577500,0.004600,1.000000,...,0.750000,13.00000,560.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.130210
50%,25000.500000,52.000000,90.00000,0.000000,1.000000,6.00000,1009.755000,954.720000,0.059800,2.000000,...,0.750000,18.00000,981.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.242322
75%,37500.250000,68.000000,135.00000,1.000000,2.000000,9.00000,1324.462500,1259.262500,0.116300,2.000000,...,0.800000,24.00000,1402.000000,2.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.414728
max,50000.000000,85.000000,180.00000,1.000000,4.000000,12.00000,3348.350000,3374.460000,0.410500,3.000000,...,0.850000,53.00000,1824.000000,10.000000,1.000000,31.000000,1.000000,1.000000,1.000000,1.000000


In [62]:
policy_df['premium_diff'] = policy_df['current_premium'] - policy_df['premium_last_year']

policy_df['premium_growth_rate'] = (
    policy_df['current_premium'] - policy_df['premium_last_year']
) / (policy_df['premium_last_year'] + 1)

policy_df['premium_per_policy'] = (
    policy_df['current_premium'] / (policy_df['num_policies'] + 1)
)

policy_df['claim_total_count'] = (
    policy_df['num_approved_claims_12m']
    + policy_df['num_rejected_claims_12m']
    + policy_df['num_pending_claims_12m']
)

policy_df['claim_approval_rate'] = (
    policy_df['num_approved_claims_12m'] / (policy_df['claim_total_count'] + 1)
)

policy_df['claim_rejection_rate'] = (
    policy_df['num_rejected_claims_12m'] / (policy_df['claim_total_count'] + 1)
)

policy_df['payment_risk_score'] = (
    policy_df['late_payment_count_12m'] + policy_df['missed_payment_flag'] * 3
)

policy_df['tenure_age_ratio'] = (
    policy_df['customer_tenure_months'] / (policy_df['age'] + 1)
)

policy_df['tenure_per_policy'] = (
    policy_df['customer_tenure_months'] / (policy_df['num_policies'] + 1)
)

policy_df['payout_to_premium_ratio'] = (
    policy_df['total_payout_amount_12m'] / (policy_df['current_premium'] + 1)
)

policy_df['contact_per_claim'] = (
    policy_df['num_contacts_12m'] / (policy_df['num_claims_12m'] + 1)
)

policy_df['renewal_risk'] = (
    policy_df['premium_change_pct'] * policy_df['num_price_increases_last_3y']
)

policy_df['price_sensitivity'] = (
    policy_df['premium_change_pct'] * policy_df['quote_requested_flag']
)

policy_df['claim_satisfaction'] = (
    policy_df['payout_ratio_12m'] / (policy_df['avg_settlement_time_days'] + 1)
)

policy_df['service_risk'] = (
    policy_df['complaint_flag'].astype(int)
    + policy_df['complaint_resolution_days'] / 30
)

policy_df['price_pressure'] = (
    policy_df['premium_change_pct'] * policy_df['num_price_increases_last_3y']
)

policy_df['coverage_value'] = (
    policy_df['coverage_amount'] / (policy_df['current_premium'] + 1)
)

policy_df['customer_dissatisfaction'] = (
    policy_df['complaint_flag'].astype(int)
    + policy_df['complaint_resolution_days'] / 30
    + policy_df['late_payment_count_12m'] / 3
)

policy_df['premium_burden'] = (
    policy_df['current_premium'] / (policy_df['coverage_amount'] + 1)
)

policy_df['engagement_drop'] = (
    policy_df['days_since_last_claim'] / (policy_df['num_contacts_12m'] + 1)
)


# =========================
# 4. feature 설정
# =========================
numeric_features = [
    'age',
    'customer_tenure_months',
    'premium_change_pct',
    'premium_diff',
    'premium_growth_rate',
    'premium_per_policy',
    'quote_requested_flag',
    'autopay_enabled',
    'late_payment_count_12m',
    'missed_payment_flag',
    'payment_risk_score',
    'coverage_downgrade_flag',
    'claim_total_count',
    'claim_approval_rate',
    'claim_rejection_rate',
    'payout_to_premium_ratio',
    'contact_per_claim',
    'tenure_age_ratio',
    'tenure_per_policy',
    'renewal_risk',
    'price_sensitivity',
    'claim_satisfaction',
    'service_risk',
    'price_pressure',
    'coverage_value',
    'customer_dissatisfaction',
    'premium_burden',
    'engagement_drop',
    'current_premium',
    'premium_last_year',
    'num_price_increases_last_3y',
    'coverage_amount',
    'premium_to_coverage_ratio',
    'payment_method_change_flag',
    'num_claims_12m',
    'num_approved_claims_12m',
    'num_rejected_claims_12m',
    'num_pending_claims_12m',
    'avg_claim_amount',
    'total_claim_amount_12m',
    'total_payout_amount_12m',
    'payout_ratio_12m',
    'avg_settlement_time_days',
    'days_since_last_claim',
    'num_contacts_12m',
    'complaint_flag',
    'complaint_resolution_days'
]

categorical_features = [
    'region_name',
    'age_band',
    'marital_status',
    'policy_type',
    'payment_frequency',
    'renewal_month'
]

all_features = numeric_features + categorical_features


In [71]:
X = policy_df[all_features]
y = policy_df["churn_flag"]



In [72]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [73]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print("scale_pos_weight:", round(scale_pos_weight, 3))

scale_pos_weight: 2.315


In [75]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [76]:
lgbm_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        class_weight="balanced"
    ))
])

xgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        gamma=0.1,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        scale_pos_weight=scale_pos_weight
    ))
])


In [77]:
X_train_processed = preprocess.fit_transform(X_train)
X_test_processed = preprocess.transform(X_test)

lgbm_base = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    class_weight="balanced"
)

xgb_base = XGBClassifier(
    n_estimators=350,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=scale_pos_weight
)

rf_base = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    random_state=42,
    class_weight="balanced"
)

stack_model = StackingClassifier(
    estimators=[
        ("lgbm", lgbm_base),
        ("xgb", xgb_base),
        ("rf", rf_base)
    ],
    final_estimator=LogisticRegression(max_iter=2000),
    cv=5,
    n_jobs=-1
)

In [78]:
def evaluate_model(model, X_train_data, X_test_data, y_train_data, y_test_data, model_name, threshold=0.50):
    model.fit(X_train_data, y_train_data)

    y_prob = model.predict_proba(X_test_data)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n===== {model_name} =====")
    print(f"Threshold = {threshold}")
    print(classification_report(y_test_data, y_pred))

    auc = roc_auc_score(y_test_data, y_prob)
    print(f"AUC: {auc:.4f}")

    return model, auc, y_prob


In [80]:
lgbm_model, lgbm_auc, lgbm_prob = evaluate_model(
    lgbm_pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    "LightGBM",
    threshold=0.50
)

[LightGBM] [Info] Number of positive: 12066, number of negative: 27934
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012143 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6150
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 82
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

C:\Users\Playdata\miniconda3\envs\tp_returns\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [81]:
xgb_model, xgb_auc, xgb_prob = evaluate_model(
    xgb_pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    "XGBoost",
    threshold=0.50
)


===== XGBoost =====
Threshold = 0.5
              precision    recall  f1-score   support

           0       0.84      0.74      0.79      6983
           1       0.53      0.66      0.59      3017

    accuracy                           0.72     10000
   macro avg       0.68      0.70      0.69     10000
weighted avg       0.74      0.72      0.73     10000

AUC: 0.7849


In [86]:
stack_trained, stack_auc, stack_prob = evaluate_model(
    stack_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test,
    "Stacking",
    threshold=0.50
)


C:\Users\Playdata\miniconda3\envs\tp_returns\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== Stacking =====
Threshold = 0.5
              precision    recall  f1-score   support

           0       0.80      0.90      0.84      6983
           1       0.67      0.46      0.55      3017

    accuracy                           0.77     10000
   macro avg       0.73      0.68      0.70     10000
weighted avg       0.76      0.77      0.75     10000

AUC: 0.7852


In [83]:
def search_best_threshold(y_true, y_prob, model_name):
    print(f"\n=== {model_name} Threshold Search ===")

    rows = []

    for t in np.arange(0.30, 0.71, 0.05):
        y_pred_t = (y_prob >= t).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred_t, average=None
        )

        rows.append([round(t, 2), round(p[1], 4), round(r[1], 4), round(f1[1], 4)])
        print(f"t={t:.2f} | P1={p[1]:.2f} R1={r[1]:.2f} F1_1={f1[1]:.2f}")

    temp_df = pd.DataFrame(rows, columns=["threshold", "precision_1", "recall_1", "f1_1"])
    best_idx = temp_df["f1_1"].idxmax()
    best_threshold = float(temp_df.loc[best_idx, "threshold"])

    print(f"Best threshold for {model_name}: {best_threshold}")
    return best_threshold, temp_df


lgbm_best_t, lgbm_t_df = search_best_threshold(y_test, lgbm_prob, "LightGBM")
xgb_best_t, xgb_t_df = search_best_threshold(y_test, xgb_prob, "XGBoost")
stack_best_t, stack_t_df = search_best_threshold(y_test, stack_prob, "Stacking")



=== LightGBM Threshold Search ===
t=0.30 | P1=0.40 R1=0.90 F1_1=0.55
t=0.35 | P1=0.43 R1=0.85 F1_1=0.57
t=0.40 | P1=0.46 R1=0.80 F1_1=0.58
t=0.45 | P1=0.50 R1=0.73 F1_1=0.59
t=0.50 | P1=0.53 R1=0.67 F1_1=0.59
t=0.55 | P1=0.57 R1=0.60 F1_1=0.59
t=0.60 | P1=0.61 R1=0.52 F1_1=0.56
t=0.65 | P1=0.66 R1=0.46 F1_1=0.55
t=0.70 | P1=0.71 R1=0.39 F1_1=0.50
Best threshold for LightGBM: 0.45

=== XGBoost Threshold Search ===
t=0.30 | P1=0.40 R1=0.89 F1_1=0.55
t=0.35 | P1=0.43 R1=0.85 F1_1=0.57
t=0.40 | P1=0.46 R1=0.80 F1_1=0.59
t=0.45 | P1=0.50 R1=0.74 F1_1=0.60
t=0.50 | P1=0.53 R1=0.66 F1_1=0.59
t=0.55 | P1=0.57 R1=0.60 F1_1=0.58
t=0.60 | P1=0.60 R1=0.53 F1_1=0.56
t=0.65 | P1=0.66 R1=0.46 F1_1=0.54
t=0.70 | P1=0.71 R1=0.39 F1_1=0.50
Best threshold for XGBoost: 0.45

=== Stacking Threshold Search ===
t=0.30 | P1=0.52 R1=0.70 F1_1=0.59
t=0.35 | P1=0.55 R1=0.63 F1_1=0.58
t=0.40 | P1=0.58 R1=0.57 F1_1=0.58
t=0.45 | P1=0.62 R1=0.51 F1_1=0.56
t=0.50 | P1=0.67 R1=0.46 F1_1=0.55
t=0.55 | P1=0.71 R1=0.40

In [85]:
model_scores = {
    "LightGBM": lgbm_auc,
    "XGBoost": xgb_auc,
    "Stacking": stack_auc
}

best_model_name = max(model_scores, key=model_scores.get)

print("\n==========================")
print("Best model by AUC:", best_model_name)
print("==========================")

if best_model_name == "LightGBM":
    best_model = lgbm_model
    best_threshold = lgbm_best_t
elif best_model_name == "XGBoost":
    best_model = xgb_model
    best_threshold = xgb_best_t
else:
    best_model = ("STACKING_SPECIAL", stack_trained)
    best_threshold = stack_best_t


Best model by AUC: Stacking
